# warcaby-zero: training on a free Kaggle GPU

AlphaZero-style self-play training for 8x8 international (Brazilian) draughts.

**Before running**

1. *Settings → Accelerator*: **GPU T4 x2** (or any GPU). *Settings → Internet*: **on** (to clone the code).
2. Set `REPO_URL` in the first code cell.
3. For a long run use **Save Version → Save & Run All (Commit)**. The notebook then runs in the
   background for up to 12 hours; you can close the browser.

**Continuing in a later session**

Everything needed to continue is saved under `runs/` in the notebook's output. To resume:
*Add Input → Notebook Output* → pick the previous version of this notebook, then run it again.
The newest checkpoint found in the inputs is copied back and training carries on from there.

Kaggle gives about 30 GPU hours a week and at most 12 hours per session, so `HOURS` below stops
the training cleanly before the session is cut off.

In [ ]:
import os

# --- settings -------------------------------------------------------------------
REPO_URL = "https://github.com/YOUR_USER/warcaby-zero.git"  # <- your repository
BRANCH = "main"
RUN_NAME = "kaggle"  # change it to start a separate run from scratch
PRESET = "kaggle"
HOURS = 11.0  # training time in this session
OVERRIDES = []  # only for a NEW run, e.g. ["simulations=120", "games_per_iteration=300"]

# These two are only changed when trying the notebook outside Kaggle.
WORK = os.environ.get("WARCABY_WORK", "/kaggle/working")
INPUT = os.environ.get("WARCABY_INPUT", "/kaggle/input")
REPO_URL = os.environ.get("WARCABY_REPO", REPO_URL)
HOURS = float(os.environ.get("WARCABY_HOURS", HOURS))
PRESET = os.environ.get("WARCABY_PRESET", PRESET)

## 1. Get the code

The code is cloned outside `/kaggle/working`, so the notebook's output holds only the run.
If the repository is private, upload it as a Kaggle Dataset instead and leave `REPO_URL` empty:
the first input folder that contains `pyproject.toml` is used.

In [ ]:
import glob
import shutil
import subprocess
import sys

CODE = os.environ.get("WARCABY_CODE", "/tmp/warcaby-zero")
shutil.rmtree(CODE, ignore_errors=True)
if REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CODE], check=True)
else:
    found = sorted(glob.glob(f"{INPUT}/**/pyproject.toml", recursive=True))
    assert found, "no REPO_URL and no dataset with the code attached"
    shutil.copytree(os.path.dirname(found[0]), CODE)

# Kaggle already has PyTorch, NumPy, TensorBoard and Matplotlib; install only the package itself.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", CODE], check=True)
print(
    subprocess.run(
        ["git", "-C", CODE, "log", "-1", "--oneline"], capture_output=True, text=True
    ).stdout
)

In [ ]:
import torch

print("PyTorch", torch.__version__, "| CPU cores:", os.cpu_count())
if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        print(f"GPU {index}:", torch.cuda.get_device_name(index))
else:
    print(
        "NO GPU - turn on the accelerator in the notebook settings, training on CPU is far too slow"
    )

## 2. Restore the previous session (if there is one)

Looks through the attached inputs for checkpoints of this run and copies the one with the most
iterations into the working directory.

In [ ]:
RUN_DIR = f"{WORK}/runs/{RUN_NAME}"


def iteration_of(checkpoint_path):
    return torch.load(checkpoint_path, map_location="cpu", weights_only=True)["iteration"]


if os.path.exists(f"{RUN_DIR}/checkpoint.pt"):
    print(
        "continuing the run already in the working directory, iteration",
        iteration_of(f"{RUN_DIR}/checkpoint.pt"),
    )
else:
    candidates = glob.glob(f"{INPUT}/**/runs/{RUN_NAME}/checkpoint.pt", recursive=True)
    if candidates:
        newest = max(candidates, key=iteration_of)
        shutil.copytree(os.path.dirname(newest), RUN_DIR)
        print(f"restored iteration {iteration_of(newest)} from {newest}")
    else:
        print("no earlier checkpoint found: starting a new run")

## 3. Train

Runs as a separate process and prints one line per iteration:
self-play statistics, losses, the arena result (was the new network accepted?) and, every few
iterations, the score against random play and minimax.

In [ ]:
command = [
    sys.executable,
    "-u",
    "-m",
    "warcaby.train",
    "--preset",
    PRESET,
    "--run-dir",
    RUN_DIR,
    "--hours",
    str(HOURS),
]
if OVERRIDES:
    command += ["--set", *OVERRIDES]
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end="", flush=True)
assert process.wait() == 0, "training failed - see the output above"

## 4. Evaluate

A tournament between a sample of the generations, random play and minimax at several depths
gives every player an Elo rating. Then the progress charts are drawn. Results land in
`runs/<name>/eval` and `runs/<name>/plots`.

In [ ]:
for step in (
    ["tournament", "--run-dir", RUN_DIR, "--generations", "8", "--games", "20"],
    ["plots", "--run-dir", RUN_DIR],
):
    subprocess.run([sys.executable, "-u", "-m", "warcaby.evaluate", *step], check=True)

In [ ]:
from IPython.display import Image, Markdown, display

display(Markdown(open(f"{RUN_DIR}/eval/summary.md").read()))
for name in ("elo", "baselines", "arena", "training"):
    path = f"{RUN_DIR}/plots/{name}.png"
    if os.path.exists(path):
        display(Image(path))

## 5. TensorBoard (interactive sessions only)

```
%load_ext tensorboard
%tensorboard --logdir /kaggle/working/runs
```

## 6. Take the results home

Download `runs/<name>` from the notebook's *Output* tab and put it into `runs/` of your local
copy of the repository. Then:

```
uv run python -m warcaby.app --run-dir runs/kaggle     # play against it
```